[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/galvezam/eia-data-pipeline/blob/main/processing/petroleum_coal_electricity_total.ipynb)


# Petroleum, Coal, Electricity & Total Energy Processing
**EIA Data Pipeline**

Datasets (sourced from `total_ingest.ipynb`):
- `petroleum_*.csv` — Weekly petroleum production by region (thousand barrels/day)
- `petroleum_movements_*.csv` — Weekly petroleum imports/exports & movements
- `coal_*.csv` — Annual coal import/export quantity and price
- `electricity_*.csv` — Monthly electricity consumption by fuel type and state
- `electricity_state_rankings_*.csv` — Annual state electricity rankings
- `electricity_net_metering_*.csv` — Annual net metering capacity and customers by state
- `electricity_generating_capacity_*.csv` — Annual generating capacity by state
- `total_energy_*.csv` — Monthly total energy by series
- `seds_*.csv` — Annual State Energy Data System (all fuels by state)

Pipeline steps: **Clean → Transform → Analyze → Upload Parquet to S3**

## 0. Environment Setup & Configuration

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Install Spark with Hadoop as the underlying scheduler
!wget -q https://downloads.apache.org/spark/spark-3.5.8/spark-3.5.8-bin-hadoop3.tgz
!tar xf spark-3.5.8-bin-hadoop3.tgz
!ls -l

os.environ["SPARK_HOME"] = "spark-3.5.8-bin-hadoop3"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

!pip install -q findspark
import findspark
findspark.init()
print("Spark environment ready.")

total 786832
drwxr-xr-x  3 root root      4096 Apr 15 03:15 __MACOSX
drwxr-xr-x 16 root root      4096 Apr  1 17:36 raw
-rw-r--r--  1 root root   2939049 Apr 15 03:15 raw.zip
drwxr-xr-x  1 root root      4096 Apr  2 13:31 sample_data
drwxr-xr-x 13 1001 1001      4096 Jan 12 04:31 spark-3.5.8-bin-hadoop3
-rw-r--r--  1 root root 401378872 Jan 12 04:34 spark-3.5.8-bin-hadoop3.tgz
-rw-r--r--  1 root root 401378872 Jan 12 04:34 spark-3.5.8-bin-hadoop3.tgz.1
Spark environment ready.


In [3]:
import sys
import glob

# AWS credentials — fill in here if using Section 5 (S3 upload)
AWS_ACCESS_KEY = ""
AWS_SECRET_KEY = ""
AWS_REGION     = "us-east-2"
BUCKET_NAME    = "cs4266-eia-big-data-bucket"

BASE = "/content/"

def latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True))
    if not files:
        raise FileNotFoundError(f"No files matched: {pattern}")
    return files[-1]

PETROLEUM_PATH            = latest(f"{BASE}/petroleum/**/*.csv")
PETROLEUM_MOVEMENTS_PATH  = latest(f"{BASE}/petroleum_movements/**/*.csv")
COAL_PATH                 = latest(f"{BASE}/coal/**/*.csv")
ELECTRICITY_PATH          = latest(f"{BASE}/electricity/**/*.csv")
ELEC_RANKINGS_PATH        = latest(f"{BASE}/electricity_state_rankings/**/*.csv")
ELEC_NET_METERING_PATH    = latest(f"{BASE}/electricity_net_metering/**/*.csv")
ELEC_CAPACITY_PATH        = latest(f"{BASE}/electricity_generating_capacity/**/*.csv")
TOTAL_ENERGY_PATH         = latest(f"{BASE}/total_energy/**/*.csv")
SEDS_PATH                 = latest(f"{BASE}/seds/**/*.csv")



# ── S3 output prefixes ──────────────────────────────────────────────────────
S3_BASE = f"s3a://{BUCKET_NAME}/processed"

S3_PETROLEUM_PRODUCTION    = f"{S3_BASE}/petroleum_production/"
S3_PETROLEUM_MOVEMENTS     = f"{S3_BASE}/petroleum_movements/"
S3_COAL_TRADE              = f"{S3_BASE}/coal_trade/"
S3_ELECTRICITY_BY_FUEL     = f"{S3_BASE}/electricity_by_fuel_state/"
S3_ELECTRICITY_RANKINGS    = f"{S3_BASE}/electricity_state_rankings/"
S3_ELECTRICITY_NET_METER   = f"{S3_BASE}/electricity_net_metering/"
S3_ELECTRICITY_CAPACITY    = f"{S3_BASE}/electricity_generating_capacity/"
S3_TOTAL_ENERGY            = f"{S3_BASE}/total_energy/"
S3_SEDS_STATE_CONSUMPTION  = f"{S3_BASE}/seds_state_consumption/"
S3_SEDS_STATE_PCT          = f"{S3_BASE}/seds_state_pct_us/"
S3_SEDS_FUEL_PIVOT         = f"{S3_BASE}/seds_fuel_pivot/"

print("Config loaded.")
print(f"  Petroleum:               {PETROLEUM_PATH}")
print(f"  Petroleum Movements:     {PETROLEUM_MOVEMENTS_PATH}")
print(f"  Coal:                    {COAL_PATH}")
print(f"  Electricity:             {ELECTRICITY_PATH}")
print(f"  Electricity Rankings:    {ELEC_RANKINGS_PATH}")
print(f"  Electricity Net Meter:   {ELEC_NET_METERING_PATH}")
print(f"  Electricity Capacity:    {ELEC_CAPACITY_PATH}")
print(f"  Total Energy:            {TOTAL_ENERGY_PATH}")
print(f"  SEDS:                    {SEDS_PATH}")
print(f"  S3 bucket:               {BUCKET_NAME}")

Config loaded.
  Petroleum:               /content/raw//petroleum/weekly/petroleum_20260401.csv
  Petroleum Movements:     /content/raw//petroleum_movements/weekly/petroleum_movements_20260401.csv
  Coal:                    /content/raw//coal/annual/coal_20260401.csv
  Electricity:             /content/raw//electricity/monthly/electricity_20260401.csv
  Electricity Rankings:    /content/raw//electricity_state_rankings/annual/electricity_state_rankings_20260401.csv
  Electricity Net Meter:   /content/raw//electricity_net_metering/annual/electricity_net_metering_20260401.csv
  Electricity Capacity:    /content/raw//electricity_generating_capacity/annual/electricity_generating_capacity_20260401.csv
  Total Energy:            /content/raw//total_energy/monthly/total_energy_20260401.csv
  SEDS:                    /content/raw//seds/annual/seds_20260401.csv
  S3 bucket:               cs4266-eia-big-data-bucket


## 1. Initialize Spark Session

In [4]:
from pyspark.sql import SparkSession

import os
print(os.environ.get('JAVA_HOME'))

builder = (
    SparkSession.builder
    .appName('EIA Petroleum, Coal, Electricity & Total Energy Processing')
    .config('spark.hadoop.validateOutputSpecs', False)
)

# Only wire up S3 if credentials are filled in
if AWS_ACCESS_KEY and AWS_SECRET_KEY and AWS_REGION:
    builder = (
        builder
        .config('spark.hadoop.fs.s3a.impl',        'org.apache.hadoop.fs.s3a.S3AFileSystem')
        .config('spark.hadoop.fs.s3a.endpoint',    f's3.{AWS_REGION}.amazonaws.com')
        .config('spark.hadoop.fs.s3a.access.key',  AWS_ACCESS_KEY)
        .config('spark.hadoop.fs.s3a.secret.key',  AWS_SECRET_KEY)
    )
    print('S3 configured.')
else:
    print('No S3 credentials — local mode only (Sections 0–4).')

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')

/usr/lib/jvm/java-8-openjdk-amd64
S3 configured.
Spark 3.5.8 ready.


## 2. Load Raw Data

In [5]:
def read_csv(path):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)   # handles quoted commas in area names
        .option("escape", '"')
        .csv(path)
    )

raw_petroleum         = read_csv(PETROLEUM_PATH)
raw_petro_movements   = read_csv(PETROLEUM_MOVEMENTS_PATH)
raw_coal              = read_csv(COAL_PATH)
raw_electricity       = read_csv(ELECTRICITY_PATH)
raw_elec_rankings     = read_csv(ELEC_RANKINGS_PATH)
raw_elec_net_meter    = read_csv(ELEC_NET_METERING_PATH)
raw_elec_capacity     = read_csv(ELEC_CAPACITY_PATH)
raw_total_energy      = read_csv(TOTAL_ENERGY_PATH)
raw_seds              = read_csv(SEDS_PATH)

datasets = {
    "Petroleum Production":          raw_petroleum,
    "Petroleum Movements":           raw_petro_movements,
    "Coal Trade":                    raw_coal,
    "Electricity by Fuel":           raw_electricity,
    "Electricity State Rankings":    raw_elec_rankings,
    "Electricity Net Metering":      raw_elec_net_meter,
    "Electricity Generating Cap":    raw_elec_capacity,
    "Total Energy":                  raw_total_energy,
    "SEDS":                          raw_seds,
}

for label, df in datasets.items():
    print(f"{label:<35} raw rows: {df.count():>10,}")

Petroleum Production                raw rows:        219
Petroleum Movements                 raw rows:        371
Coal Trade                          raw rows:      3,931
Electricity by Fuel                 raw rows:    101,743
Electricity State Rankings          raw rows:      1,727
Electricity Net Metering            raw rows:        521
Electricity Generating Cap          raw rows:      4,727
Total Energy                        raw rows:     23,127
SEDS                                raw rows:     62,149


In [6]:
# Inspect schemas for all raw datasets
for label, df in datasets.items():
    print(f"\n=== {label} Schema ===")
    df.printSchema()


=== Petroleum Production Schema ===
root
 |-- period: string (nullable = true)
 |-- duoarea: string (nullable = true)
 |-- area-name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- product-name: string (nullable = true)
 |-- process: string (nullable = true)
 |-- process-name: string (nullable = true)
 |-- series: string (nullable = true)
 |-- series-description: string (nullable = true)
 |-- value: string (nullable = true)
 |-- units: string (nullable = true)


=== Petroleum Movements Schema ===
root
 |-- period: string (nullable = true)
 |-- duoarea: string (nullable = true)
 |-- area-name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- product-name: string (nullable = true)
 |-- process: string (nullable = true)
 |-- process-name: string (nullable = true)
 |-- series: string (nullable = true)
 |-- series-description: string (nullable = true)
 |-- value: string (nullable = true)
 |-- units: string (nullable = true)


=== Coal Trade Schema 

## 3. Cleaning

### 3a. Petroleum Production

In [10]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

petroleum_clean = (
    raw_petroleum
    .filter(F.col("value").isNotNull())
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM-dd"))
    .withColumn("value_kbd", F.col("value").cast("double"))
    .withColumnRenamed("area-name", "area_name")
    .withColumn("area_name",    F.trim(F.col("area_name")))
    .withColumn("product_name", F.trim(F.col("product-name")))
    .withColumn("process_name", F.trim(F.col("process-name")))
    .withColumn("series_description", F.trim(F.col("series-description")))
    .select(
        "period", "duoarea", "area_name",
        F.col("product").alias("product_id"),
        "product_name",
        F.col("process").alias("process_id"),
        "process_name",
        F.col("series").alias("series_id"),
        "series_description",
        "value_kbd", "units"
    )
    .dropDuplicates()
    .orderBy("period", "duoarea", "product_id")
)

print(f"Petroleum Production cleaned rows: {petroleum_clean.count():,}")
petroleum_clean.show(5)

Petroleum Production cleaned rows: 110
+----------+-------+---------+----------+--------------------+----------+--------------------+-----------+--------------------+---------+------+
|    period|duoarea|area_name|product_id|        product_name|process_id|        process_name|  series_id|  series_description|value_kbd| units|
+----------+-------+---------+----------+--------------------+----------+--------------------+-----------+--------------------+---------+------+
|      NULL|duoarea|area-name|   product|        product-name|   process|        process-name|     series|  series-description|     NULL| units|
|2025-08-01|    NUS|     U.S.|      EPD0| Distillate Fuel Oil|       YPR|Refinery and Blen...|   WDIRPUS2|U.S. Refiner and ...|   5105.0|MBBL/D|
|2025-08-01|    NUS|     U.S.|    EPD00H|Distillate Fuel O...|       YPR|Refinery and Blen...|   WDGRPUS2|U.S.  Refiner and...|     70.0|MBBL/D|
|2025-08-01|    NUS|     U.S.|    EPDM10|Distillate Fuel O...|       YPR|Refinery and Blen.

### 3b. Petroleum Movements (Imports / Exports)

In [11]:
# Map process codes to readable labels for petroleum movements
PETRO_MOVEMENT_LABELS = {
    "EEX": "Exports",
    "EIM": "Imports",
    "ENT": "Net Imports",
    "STK": "Stock Change",
    "VUA": "Supply Adjustment",
    "YPA": "Net Production",
}
petro_process_map = F.create_map(
    *[item for pair in
      [(F.lit(k), F.lit(v)) for k, v in PETRO_MOVEMENT_LABELS.items()]
      for item in pair]
)

petro_movements_clean = (
    raw_petro_movements
    .filter(F.col("value").isNotNull())
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM-dd"))
    .withColumn("value_kbd", F.col("value").cast("double"))
    .withColumn(
        "process_label",
        F.coalesce(petro_process_map[F.col("process")], F.trim(F.col("process-name")))
    )
    .withColumnRenamed("area-name", "area_name")
    .withColumn("area_name",    F.trim(F.col("area_name")))
    .withColumn("product_name", F.trim(F.col("product-name")))
    .withColumn("series_description", F.trim(F.col("series-description")))
    .select(
        "period", "duoarea", "area_name",
        F.col("product").alias("product_id"),
        "product_name",
        F.col("process").alias("process_id"),
        "process_label",
        F.col("series").alias("series_id"),
        "series_description",
        "value_kbd", "units"
    )
    .dropDuplicates()
    .orderBy("period", "duoarea", "product_id")
)

print(f"Petroleum Movements cleaned rows: {petro_movements_clean.count():,}")
petro_movements_clean.show(5)

Petroleum Movements cleaned rows: 186
+----------+-------+---------+----------+--------------------+----------+-------------+---------+--------------------+---------+------+
|    period|duoarea|area_name|product_id|        product_name|process_id|process_label|series_id|  series_description|value_kbd| units|
+----------+-------+---------+----------+--------------------+----------+-------------+---------+--------------------+---------+------+
|      NULL|duoarea|area-name|   product|        product-name|   process| process-name|   series|  series-description|     NULL| units|
|2025-08-01|NUS-Z00|     U.S.|      EP00|Crude Oil and Pet...|       IM0|      Imports| WTTIMUS2|U.S. Imports of C...|   7477.0|MBBL/D|
|2025-08-01|NUS-Z00|     U.S.|      EP00|Crude Oil and Pet...|       IMN|  Net Imports| WTTNTUS2|U.S. Net Imports ...|  -3186.0|MBBL/D|
|2025-08-01|NUS-Z00|     U.S.|      EP00|Crude Oil and Pet...|       EEX|      Exports| WTTEXUS2|U.S. Exports of C...|  10663.0|MBBL/D|
|2025-08-0

### 3c. Coal Trade

In [12]:
coal_clean = (
    raw_coal
    .filter(F.col("quantity").isNotNull() | F.col("price").isNotNull())
    .withColumn("year",               F.col("period").cast(IntegerType()))
    .withColumn("quantity_short_tons", F.col("quantity").cast("double"))
    .withColumn("price_usd_per_ton",   F.col("price").cast("double"))
    .withColumn("export_import_type",  F.trim(F.col("exportImportType")))
    .withColumn("coal_rank_id",        F.trim(F.col("coalRankId")))
    .withColumn("coal_rank_desc",      F.trim(F.col("coalRankDescription")))
    .withColumn("country_id",          F.trim(F.col("countryId")))
    .withColumn("country_desc",        F.trim(F.col("countryDescription")))
    .withColumn("customs_district_id", F.trim(F.col("customsDistrictId")))
    .withColumn("customs_district",    F.trim(F.col("customsDistrictDescription")))
    # Exclude aggregate "All" country rows from state-level breakdowns
    .filter(F.col("country_id") != "TOT")
    .filter(
        F.col("quantity_short_tons").isNotNull() |
        F.col("price_usd_per_ton").isNotNull()
    )
    .select(
        "year", "export_import_type",
        "coal_rank_id", "coal_rank_desc",
        "country_id", "country_desc",
        "customs_district_id", "customs_district",
        "quantity_short_tons", "price_usd_per_ton"
    )
    .dropDuplicates()
    .orderBy("year", "export_import_type", "country_id")
)

print(f"Coal cleaned rows: {coal_clean.count():,}")
coal_clean.show(5)

# ── US totals by direction + coal rank ────────────────────────────────────
us_coal_total = (
    coal_clean
    .groupBy("year", "export_import_type", "coal_rank_id")
    .agg(
        F.sum("quantity_short_tons").alias("us_total_short_tons"),
        F.avg("price_usd_per_ton").alias("us_avg_price_usd_per_ton")
    )
)

# ── By country: volume, avg price, % of US total ──────────────────────────
coal_by_country = (
    coal_clean
    .groupBy("year", "export_import_type", "coal_rank_id", "coal_rank_desc",
             "country_id", "country_desc")
    .agg(
        F.sum("quantity_short_tons").alias("total_short_tons"),
        F.avg("price_usd_per_ton").alias("avg_price_usd_per_ton")
    )
    .join(us_coal_total, on=["year", "export_import_type", "coal_rank_id"], how="left")
    .withColumn(
        "pct_us_total",
        F.round(F.col("total_short_tons") / F.col("us_total_short_tons") * 100, 4)
    )
    .orderBy("year", "export_import_type", F.col("total_short_tons").desc())
)

print(f"Coal by country rows: {coal_by_country.count():,}")
coal_by_country.show(10)

# ── By customs district: which ports handle the most volume ───────────────
coal_by_district = (
    coal_clean
    .groupBy("year", "export_import_type", "coal_rank_id", "coal_rank_desc",
             "customs_district_id", "customs_district")
    .agg(
        F.sum("quantity_short_tons").alias("total_short_tons"),
        F.avg("price_usd_per_ton").alias("avg_price_usd_per_ton"),
        F.countDistinct("country_id").alias("num_countries")
    )
    .join(us_coal_total, on=["year", "export_import_type", "coal_rank_id"], how="left")
    .withColumn(
        "pct_us_total",
        F.round(F.col("total_short_tons") / F.col("us_total_short_tons") * 100, 4)
    )
    .orderBy("year", "export_import_type", F.col("total_short_tons").desc())
)

print(f"Coal by customs district rows: {coal_by_district.count():,}")
coal_by_district.show(10)

# ── Coal rank breakdown (Bituminous, Coke, etc.) by direction + year ──────
coal_by_rank = (
    coal_clean
    .groupBy("year", "export_import_type", "coal_rank_id", "coal_rank_desc")
    .agg(
        F.sum("quantity_short_tons").alias("total_short_tons"),
        F.avg("price_usd_per_ton").alias("avg_price_usd_per_ton"),
        F.min("price_usd_per_ton").alias("min_price_usd_per_ton"),
        F.max("price_usd_per_ton").alias("max_price_usd_per_ton"),
        F.countDistinct("country_id").alias("num_countries"),
        F.countDistinct("customs_district_id").alias("num_districts")
    )
    .orderBy("year", "export_import_type", F.col("total_short_tons").desc())
)

print(f"Coal by rank rows: {coal_by_rank.count():,}")
coal_by_rank.show(10)

# ── YoY change per country ────────────────────────────────────────────────
country_year_window = Window.partitionBy(
    "export_import_type", "coal_rank_id", "country_id"
).orderBy("year")

coal_country_yoy = (
    coal_by_country
    .withColumn("yoy_change_short_tons",
        F.col("total_short_tons") - F.lag("total_short_tons", 1).over(country_year_window)
    )
    .withColumn("yoy_change_pct",
        F.round(
            F.col("yoy_change_short_tons") / F.lag("total_short_tons", 1).over(country_year_window) * 100,
            2
        )
    )
    .orderBy("year", "export_import_type", F.col("total_short_tons").desc())
)

print(f"Coal country YoY rows: {coal_country_yoy.count():,}")
coal_country_yoy.show(10)

# ── Pivot: one row per year-direction, one column per coal rank ───────────
coal_rank_ids = [
    r["coal_rank_id"] for r in
    coal_clean.select("coal_rank_id").distinct().orderBy("coal_rank_id").collect()
]

coal_rank_pivot = (
    coal_clean
    .groupBy("year", "export_import_type")
    .pivot("coal_rank_id", coal_rank_ids)
    .agg(F.sum("quantity_short_tons"))
    .orderBy("year", "export_import_type")
)

print(f"Coal rank pivot rows: {coal_rank_pivot.count():,}")
coal_rank_pivot.show(10)

Coal cleaned rows: 1,648
+----+------------------+------------+--------------+----------+--------------------+-------------------+--------------------+-------------------+-----------------+
|year|export_import_type|coal_rank_id|coal_rank_desc|country_id|        country_desc|customs_district_id|    customs_district|quantity_short_tons|price_usd_per_ton|
+----+------------------+------------+--------------+----------+--------------------+-------------------+--------------------+-------------------+-----------------+
|2024|           Exports|         STM|    Steam Coal|        AE|United Arab Emirates|              NO_LA|     New Orleans, LA|            59098.0|            53.71|
|2024|           Exports|         STM|    Steam Coal|        AE|United Arab Emirates|              HG_TX|Houston-Galveston...|                4.0|          1272.75|
|2024|           Exports|         TOT|           All|        AE|United Arab Emirates|              NY_NY|   New York City, NY|               45.0|    

### 3d. Electricity Operations by Fuel Type

In [13]:
# Map fuel type IDs to readable labels
FUEL_TYPE_LABELS = {
    "COL": "Coal",
    "NG":  "Natural Gas",
    "NUC": "Nuclear",
    "HYC": "Conventional Hydroelectric",
    "WND": "Wind",
    "SUN": "Solar",
    "GEO": "Geothermal",
    "ALL": "All Fuels",
    "AOR": "All Renewables",
}
fuel_map = F.create_map(
    *[item for pair in
      [(F.lit(k), F.lit(v)) for k, v in FUEL_TYPE_LABELS.items()]
      for item in pair]
)

electricity_clean = (
    raw_electricity
    # ── Remove rows where both consumption columns are null ────────────────
    .filter(
        F.col("consumption-for-eg").isNotNull() |
        F.col("consumption-for-eg-btu").isNotNull()
    )
    # ── Keep only known fuel types (COL, NG, NUC, etc.) ───────────────────
    .filter(F.col("fueltypeid").isin(list(FUEL_TYPE_LABELS.keys())))
    # ── Keep state-level rows: stateid is a 2-letter state code ────────────
    # .filter(F.col("stateid").rlike(r"^[A-Z]{2}$"))
    # ── Parse monthly period to date ───────────────────────────────────────
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM"))
    # ── Cast metric columns ────────────────────────────────────────────────
    .withColumn("consumption_thousand_mwh",
        F.col("consumption-for-eg").cast("double"))
    .withColumn("consumption_btu",
        F.col("consumption-for-eg-btu").cast("double"))
    # ── Add human-readable fuel label ──────────────────────────────────────
    .withColumn(
        "fuel_type_label",
        F.coalesce(fuel_map[F.col("fueltypeid")], F.col("fueltypeid"))
    )
    # ── Trim and rename ────────────────────────────────────────────────────
    .withColumn("state_name", F.trim(F.col("stateDescription")))
    # ── Select relevant columns ────────────────────────────────────────────
    .select(
        "period",
        # F.col("stateid").alias("state_id"),
        "state_name",
        F.col("fueltypeid").alias("fuel_type_id"),
        "fuel_type_label",
        "consumption_thousand_mwh",
        "consumption_btu"
    )
    .dropDuplicates()
    .orderBy("period", "fuel_type_id")
)

print(f"Electricity cleaned rows: {electricity_clean.count():,}")
electricity_clean.show(5)

Electricity cleaned rows: 32,983
+----------+-----------+------------+---------------+------------------------+---------------+
|    period| state_name|fuel_type_id|fuel_type_label|consumption_thousand_mwh|consumption_btu|
+----------+-----------+------------+---------------+------------------------+---------------+
|2025-01-01|Connecticut|         ALL|      All Fuels|                     0.0|        0.03916|
|2025-01-01|   Virginia|         ALL|      All Fuels|                     0.0|       85.83545|
|2025-01-01|    Florida|         ALL|      All Fuels|                     0.0|        1.68214|
|2025-01-01|   Virginia|         ALL|      All Fuels|                     0.0|        1.62913|
|2025-01-01|  Louisiana|         ALL|      All Fuels|                     0.0|       54.90441|
+----------+-----------+------------+---------------+------------------------+---------------+
only showing top 5 rows



### 3e. Electricity State Rankings

In [14]:
RANKING_COLS = [
    "average-retail-price-rank",
    "carbon-dioxide-rank",
    "direct-use-rank",
    "fsp-sales-rank",
    "generation-elect-utils-rank",
    "net-generation-rank",
    "nitrogen-oxide-rank",
    "sulfer-dioxide-rank",
]

# Cast and rename all rank columns
elec_rankings_clean = raw_elec_rankings
for col in RANKING_COLS:
    safe_name = col.replace("-", "_")
    elec_rankings_clean = elec_rankings_clean.withColumn(
        safe_name, F.col(f"`{col}`").cast(IntegerType())
    )

elec_rankings_clean = (
    elec_rankings_clean
    # ── Drop rows where all rank columns are null ──────────────────────────
    .filter(
        F.greatest(*[F.col(c.replace("-", "_")).isNotNull()
                     for c in RANKING_COLS])
    )
    .withColumn("year", F.col("period").cast(IntegerType()))
    .withColumn("state_id",   F.trim(F.col("stateID")))
    .withColumn("state_name", F.trim(F.col("stateDescription")))
    .select(
        "year", "state_id", "state_name",
        *[c.replace("-", "_") for c in RANKING_COLS]
    )
    .dropDuplicates()
    .orderBy("year", "state_id")
)

print(f"Electricity Rankings cleaned rows: {elec_rankings_clean.count():,}")
elec_rankings_clean.show(5)

Electricity Rankings cleaned rows: 846
+----+--------+----------+-------------------------+-------------------+---------------+--------------+---------------------------+-------------------+-------------------+-------------------+
|year|state_id|state_name|average_retail_price_rank|carbon_dioxide_rank|direct_use_rank|fsp_sales_rank|generation_elect_utils_rank|net_generation_rank|nitrogen_oxide_rank|sulfer_dioxide_rank|
+----+--------+----------+-------------------------+-------------------+---------------+--------------+---------------------------+-------------------+-------------------+-------------------+
|2008|      AK|    Alaska|                        6|                 46|             37|            48|                         39|                 50|                 41|                 48|
|2008|      AL|   Alabama|                       27|                 10|              6|            13|                          2|                  7|                 10|                  6|
|

### 3f. Electricity Net Metering

In [15]:
elec_net_meter_clean = (
    raw_elec_net_meter
    # ── Drop rows where both metrics are null ──────────────────────────────
    .filter(
        F.col("capacity").isNotNull() |
        F.col("customers").isNotNull()
    )
    .withColumn("year",           F.col("period").cast(IntegerType()))
    .withColumn("capacity_mw",    F.col("capacity").cast("double"))
    .withColumn("customers_count",F.col("customers").cast("double").cast(IntegerType()))
    .withColumn("state_id",   F.trim(F.col("state")))
    .withColumn("state_name", F.trim(F.col("stateName")))
    # ── Keep state-level rows only ─────────────────────────────────────────
    .filter(F.col("state_id").rlike(r"^[A-Z]{2}$"))
    .select(
        "year", "state_id", "state_name",
        "sector", "sectorName",
        "capacity_mw", "customers_count"
    )
    .dropDuplicates()
    .orderBy("year", "state_id")
)

print(f"Electricity Net Metering cleaned rows: {elec_net_meter_clean.count():,}")
elec_net_meter_clean.show(5)

Electricity Net Metering cleaned rows: 260
+----+--------+----------+------+--------------+-----------+---------------+
|year|state_id|state_name|sector|    sectorName|capacity_mw|customers_count|
+----+--------+----------+------+--------------+-----------+---------------+
|2024|      AK|    Alaska|   COM|    commercial|      2.646|            242|
|2024|      AK|    Alaska|   IND|    industrial|      0.121|              9|
|2024|      AK|    Alaska|   TRN|transportation|        0.0|              0|
|2024|      AK|    Alaska|   RES|   residential|     14.421|           2771|
|2024|      AK|    Alaska|   TOT|   all sectors|     17.188|           3022|
+----+--------+----------+------+--------------+-----------+---------------+
only showing top 5 rows



### 3g. Electricity Generating Capacity

In [16]:
elec_capacity_clean = (
    raw_elec_capacity
    .filter(F.col("capability").isNotNull())
    .withColumn("year",           F.col("period").cast(IntegerType()))
    .withColumn("capability_mw",  F.col("capability").cast("double"))
    .withColumn("state_id",   F.trim(F.col("stateID")))
    .withColumn("state_name", F.trim(F.col("stateDescription")))
    # ── Keep state-level rows only ─────────────────────────────────────────
    .filter(F.col("state_id").rlike(r"^[A-Z]{2}$"))
    .select(
        "year", "state_id", "state_name",
        "producertypeid",
        F.col("producerTypeDescription").alias("producer_type"),
        "energysourceid",
        F.col("energySourceDescription").alias("energy_source"),
        "capability_mw"
    )
    .dropDuplicates()
    .orderBy("year", "state_id", "energysourceid")
)

print(f"Electricity Generating Capacity cleaned rows: {elec_capacity_clean.count():,}")
elec_capacity_clean.show(5)

Electricity Generating Capacity cleaned rows: 2,363
+----+--------+----------+--------------+--------------------+--------------+-------------+-------------+
|year|state_id|state_name|producertypeid|       producer_type|energysourceid|energy_source|capability_mw|
+----+--------+----------+--------------+--------------------+--------------+-------------+-------------+
|2024|      AK|    Alaska|           TOT|         All sectors|           ALL|          All|       2853.4|
|2024|      AK|    Alaska|            EU|  Electric Utilities|           ALL|          All|       2638.9|
|2024|      AK|    Alaska|           IPP|Independent Power...|           ALL|          All|        214.5|
|2024|      AK|    Alaska|           TOT|         All sectors|           BAT|      Battery|        131.7|
|2024|      AK|    Alaska|            EU|  Electric Utilities|           BAT|      Battery|        131.7|
+----+--------+----------+--------------+--------------------+--------------+-------------+---------

### 3h. Total Energy

In [17]:
total_energy_clean = (
    raw_total_energy
    .filter(F.col("value").isNotNull())
    .withColumn("period", F.to_date(F.col("period"), "yyyy-MM"))
    .withColumn("value_quad_btu", F.col("value").cast("double"))
    # ── Trim description fields ────────────────────────────────────────────
    .withColumn("series-description",
        F.trim(F.col("seriesDescription"))
    )
    .select(
        "period",
        F.col("msn").alias("series_code"),
        F.col("series-description").alias("series_description"),
        "value_quad_btu", "unit"
    )
    .dropDuplicates()
    .orderBy("period", "series_code")
)

print(f"Total Energy cleaned rows: {total_energy_clean.count():,}")
total_energy_clean.show(5)

Total Energy cleaned rows: 11,564
+----------+-----------+--------------------+--------------+--------------------+
|    period|series_code|  series_description|value_quad_btu|                unit|
+----------+-----------+--------------------+--------------+--------------------+
|      NULL|        msn|   seriesDescription|          NULL|                unit|
|2025-01-01|    ARICBUS|Asphalt and Road ...|        46.036|        Trillion Btu|
|2025-01-01|    ARICPUS|Asphalt and Road ...|       223.783|Thousand Barrels ...|
|2025-01-01|    ARNFBUS|Asphalt & Road Oi...|      0.046036|     Quadrillion Btu|
|2025-01-01|    ARNFPUS|Asphalt & Road Oi...|       223.783|Thousand Barrels ...|
+----------+-----------+--------------------+--------------+--------------------+
only showing top 5 rows



### 3i. SEDS — State Energy Data System

In [18]:
# SEDS msn (series) codes that represent state-level end-use consumption
# by major fuel: Coal (CO), Petroleum (PA), Natural Gas (NG), Electricity (ES),
# Nuclear (NU), Renewables (RE), Total (TE).
# We keep the 'C' suffix codes = consumption in physical units (million BTU).
SEDS_FUEL_PREFIXES = {
    "CO": "Coal",
    "PA": "Petroleum",
    "NG": "Natural Gas",
    "ES": "Electricity",
    "NU": "Nuclear",
    "RE": "Renewables",
    "TE": "Total Energy",
    "GE": "Geothermal",
    "HY": "Hydroelectric",
    "SO": "Solar",
    "WY": "Wind",
    "WD": "Wood",
    "WS": "Waste",
}
fuel_prefix_map = F.create_map(
    *[item for pair in
      [(F.lit(k), F.lit(v)) for k, v in SEDS_FUEL_PREFIXES.items()]
      for item in pair]
)

seds_clean = (
    raw_seds
    .filter(F.col("value").isNotNull())
    # Keep state-level rows only (2-letter stateId, exclude US)
    .filter(F.col("stateId").rlike(r"^[A-Z]{2}$"))
    .filter(F.col("stateId") != "US")
    .withColumn("year", F.col("period").cast(IntegerType()))
    .withColumn("value_d", F.col("value").cast("double"))
    .filter(F.col("value_d").isNotNull())
    # Extract fuel prefix from seriesId (first 2 chars)
    .withColumn("fuel_prefix", F.substring(F.col("seriesId"), 1, 2))
    # Extract sector code from seriesId (chars 3-4)
    .withColumn("sector_code", F.substring(F.col("seriesId"), 3, 2))
    # Extract unit type from seriesId (last char: B=Btu, P=physical, C=cost)
    .withColumn("unit_type", F.substring(F.col("seriesId"), 5, 1))
    .withColumn("state_name", F.trim(F.col("stateDescription")))
    .withColumn("series_description", F.trim(F.col("seriesDescription")))
    .select(
        "year",
        F.col("stateId").alias("state_id"),
        "state_name",
        F.col("seriesId").alias("series_code"),
        "series_description",
        "fuel_prefix",
        "sector_code",
        "unit_type",
        F.col("value_d").alias("value"),
        "unit"
    )
    .dropDuplicates()
    .orderBy("year", "state_id", "series_code")
)

print(f"SEDS cleaned rows: {seds_clean.count():,}")
seds_clean.show(5)

# ── State-year-fuel aggregation ────────────────────────────────────────────
# Keep only Btu rows (unit_type == "B") for consistent cross-fuel comparison
seds_state_fuel = (
    seds_clean
    .filter(F.col("unit_type") == "B")
    .groupBy("year", "state_id", "state_name", "fuel_prefix")
    .agg(F.sum("value").alias("total_btu"))
)

# ── US annual totals by fuel prefix ───────────────────────────────────────
us_seds_total = (
    seds_state_fuel
    .groupBy("year", "fuel_prefix")
    .agg(F.sum("total_btu").alias("us_total_btu"))
)

# ── State consumption with % of US per fuel ───────────────────────────────
seds_state_consumption = (
    seds_state_fuel
    .join(us_seds_total, on=["year", "fuel_prefix"], how="left")
    .withColumn(
        "pct_us_fuel",
        F.round(F.col("total_btu") / F.col("us_total_btu") * 100, 4)
    )
    .orderBy("year", "state_id", "fuel_prefix")
)

print(f"SEDS state consumption rows: {seds_state_consumption.count():,}")
seds_state_consumption.show(10)

# ── US total across all fuels (for overall state share) ───────────────────
us_seds_all = (
    seds_state_fuel
    .groupBy("year")
    .agg(F.sum("total_btu").alias("us_all_btu"))
)

state_year_window = Window.partitionBy("state_id").orderBy("year")

# ── State % of total US energy + YoY change ──────────────────────────────
seds_state_pct_us = (
    seds_state_fuel
    .groupBy("year", "state_id", "state_name")
    .agg(F.sum("total_btu").alias("total_btu"))
    .join(us_seds_all, on="year", how="left")
    .withColumn(
        "pct_us_total",
        F.round(F.col("total_btu") / F.col("us_all_btu") * 100, 4)
    )
    .withColumn("yoy_change_btu",
        F.col("total_btu") - F.lag("total_btu", 1).over(state_year_window)
    )
    .withColumn("yoy_change_pct",
        F.round(
            F.col("yoy_change_btu") / F.lag("total_btu", 1).over(state_year_window) * 100,
            2
        )
    )
    .orderBy("year", F.col("total_btu").desc())
)

print(f"SEDS state % of US rows: {seds_state_pct_us.count():,}")
seds_state_pct_us.show(10)

# ── Fuel pivot: one row per state-year, one column per fuel prefix ─────────
# Get all distinct fuel prefixes dynamically
fuel_prefixes = [
    r["fuel_prefix"] for r in
    seds_state_fuel.select("fuel_prefix").distinct().orderBy("fuel_prefix").collect()
]

seds_fuel_pivot = (
    seds_state_fuel
    .groupBy("year", "state_id", "state_name")
    .pivot("fuel_prefix", fuel_prefixes)
    .agg(F.first("total_btu"))
    .orderBy("year", "state_id")
)

print(f"SEDS fuel pivot rows: {seds_fuel_pivot.count():,}")
seds_fuel_pivot.show(5)

SEDS cleaned rows: 30,447
+----+--------+----------+-----------+--------------------+-----------+-----------+---------+------+--------------------+
|year|state_id|state_name|series_code|  series_description|fuel_prefix|sector_code|unit_type| value|                unit|
+----+--------+----------+-----------+--------------------+-----------+-----------+---------+------+--------------------+
|2024|      AK|    Alaska|      AVACB|Aviation gasoline...|         AV|         AC|        B|1096.0|         Billion Btu|
|2024|      AK|    Alaska|      AVACD|Aviation gasoline...|         AV|         AC|        D| 31.81|Dollars per milli...|
|2024|      AK|    Alaska|      AVACE|Aviation gasoline...|         AV|         AC|        E| 0.076|Million metric to...|
|2024|      AK|    Alaska|      AVACP|Aviation gasoline...|         AV|         AC|        P| 217.0|    Thousand barrels|
|2024|      AK|    Alaska|      AVACV|Aviation gasoline...|         AV|         AC|        V|  34.8|     Million dollars

## 4. Analytics

### 4a. Electricity: State Consumption by Fuel Over Time & % of US Total

In [19]:
# ── US monthly total per fuel type ────────────────────────────────────────
us_elec_total = (
    electricity_clean
    .filter(F.col("fuel_type_id") == "ALL")
    .groupBy("period")
    .agg(F.sum("consumption_thousand_mwh").alias("us_total_thousand_mwh"))
)

# ── State consumption across all fuels + % of US total ───────────────────
elec_state_total = (
    electricity_clean
    .filter(F.col("fuel_type_id") == "ALL")
    .groupBy("period", "state_name")
    .agg(F.sum("consumption_thousand_mwh").alias("state_total_thousand_mwh"))
    .join(us_elec_total, on="period", how="left")
    .withColumn(
        "pct_us_consumption",
        F.round(
            F.col("state_total_thousand_mwh") / F.col("us_total_thousand_mwh") * 100,
            4
        )
    )
    .select(
        "period", "state_name",
        "state_total_thousand_mwh", "us_total_thousand_mwh", "pct_us_consumption"
    )
    .orderBy("period", F.col("state_total_thousand_mwh").desc())
)

print(f"Electricity state consumption rows: {elec_state_total.count():,}")
elec_state_total.show(10)

Electricity state consumption rows: 768
+----------+------------------+------------------------+---------------------+------------------+
|    period|        state_name|state_total_thousand_mwh|us_total_thousand_mwh|pct_us_consumption|
+----------+------------------+------------------------+---------------------+------------------+
|2025-01-01|          Colorado|                     0.0|                  0.0|              NULL|
|2025-01-01|East South Central|                     0.0|                  0.0|              NULL|
|2025-01-01|        U.S. Total|                     0.0|                  0.0|              NULL|
|2025-01-01|         Minnesota|                     0.0|                  0.0|              NULL|
|2025-01-01|       Connecticut|                     0.0|                  0.0|              NULL|
|2025-01-01|         Wisconsin|                     0.0|                  0.0|              NULL|
|2025-01-01|    South Carolina|                     0.0|                  0.0|

### 4b. Electricity: State Consumption by Fuel Type (Monthly)

In [20]:
# ── US monthly total per fuel type ────────────────────────────────────────
us_elec_by_fuel = (
    electricity_clean
    .filter(F.col("fuel_type_id") != "ALL")   # exclude the aggregate row
    .groupBy("period", "fuel_type_id", "fuel_type_label")
    .agg(F.sum("consumption_thousand_mwh").alias("us_fuel_thousand_mwh"))
)

# ── State-fuel breakdown with % share of US fuel consumption ─────────────
elec_state_by_fuel = (
    electricity_clean
    .filter(F.col("fuel_type_id") != "ALL")
    .join(
        us_elec_by_fuel.select(
            "period", "fuel_type_id", "us_fuel_thousand_mwh"
        ),
        on=["period", "fuel_type_id"],
        how="left"
    )
    .withColumn(
        "pct_us_fuel_consumption",
        F.round(
            F.col("consumption_thousand_mwh") / F.col("us_fuel_thousand_mwh") * 100,
            4
        )
    )
    .select(
        "period", "state_name",
        "fuel_type_id", "fuel_type_label",
        "consumption_thousand_mwh", "consumption_btu",
        "us_fuel_thousand_mwh", "pct_us_fuel_consumption"
    )
    .orderBy("period", "state_name", "fuel_type_id")
)

print(f"Electricity by fuel & state rows: {elec_state_by_fuel.count():,}")
elec_state_by_fuel.show(10)

Electricity by fuel & state rows: 25,029
+----------+----------+------------+--------------------+------------------------+---------------+--------------------+-----------------------+
|    period|state_name|fuel_type_id|     fuel_type_label|consumption_thousand_mwh|consumption_btu|us_fuel_thousand_mwh|pct_us_fuel_consumption|
+----------+----------+------------+--------------------+------------------------+---------------+--------------------+-----------------------+
|2025-01-01|   Alabama|         AOR|      All Renewables|                     0.0|        0.01861|                 0.0|                   NULL|
|2025-01-01|   Alabama|         AOR|      All Renewables|                     0.0|        1.79246|                 0.0|                   NULL|
|2025-01-01|   Alabama|         AOR|      All Renewables|                     0.0|        0.29148|                 0.0|                   NULL|
|2025-01-01|   Alabama|         AOR|      All Renewables|                     0.0|        2.083

### 4c. Coal: Import/Export Summary by Country & Direction (Annual)

In [21]:
# ── US annual totals by direction + coal rank ─────────────────────────────
us_coal_total = (
    coal_clean
    .groupBy("year", "export_import_type", "coal_rank_id")
    .agg(
        F.sum("quantity_short_tons").alias("us_total_short_tons"),
        F.avg("price_usd_per_ton").alias("us_avg_price_usd_per_ton")
    )
)

# ── Country breakdown with % of US total ─────────────────────────────────
coal_trade_summary = (
    coal_clean
    .groupBy("year", "export_import_type", "coal_rank_id", "coal_rank_desc",
             "country_id", "country_desc",
             "customs_district_id", "customs_district")
    .agg(
        F.sum("quantity_short_tons").alias("total_short_tons"),
        F.avg("price_usd_per_ton").alias("avg_price_usd_per_ton"),
        F.min("price_usd_per_ton").alias("min_price_usd_per_ton"),
        F.max("price_usd_per_ton").alias("max_price_usd_per_ton"),
    )
    .join(us_coal_total, on=["year", "export_import_type", "coal_rank_id"], how="left")
    .withColumn(
        "pct_us_direction",
        F.round(F.col("total_short_tons") / F.col("us_total_short_tons") * 100, 4)
    )
    .select(
        "year", "export_import_type",
        "coal_rank_id", "coal_rank_desc",
        "country_id", "country_desc",
        "customs_district_id", "customs_district",
        "total_short_tons", "avg_price_usd_per_ton",
        "min_price_usd_per_ton", "max_price_usd_per_ton",
        "us_total_short_tons", "us_avg_price_usd_per_ton", "pct_us_direction"
    )
    .orderBy("year", "export_import_type", F.col("total_short_tons").desc())
)

print(f"Coal trade summary rows: {coal_trade_summary.count():,}")
coal_trade_summary.show(10)

Coal trade summary rows: 1,648
+----+------------------+------------+--------------+----------+------------+-------------------+----------------+----------------+---------------------+---------------------+---------------------+-------------------+------------------------+----------------+
|year|export_import_type|coal_rank_id|coal_rank_desc|country_id|country_desc|customs_district_id|customs_district|total_short_tons|avg_price_usd_per_ton|min_price_usd_per_ton|max_price_usd_per_ton|us_total_short_tons|us_avg_price_usd_per_ton|pct_us_direction|
+----+------------------+------------+--------------+----------+------------+-------------------+----------------+----------------+---------------------+---------------------+---------------------+-------------------+------------------------+----------------+
|2024|           Exports|         TOT|           All|        IN|       India|                TOT|           Total|     2.4946411E7|               122.28|               122.28|              

### 4d. SEDS: State Energy Consumption Over Time by Fuel Category (Annual)

In [22]:
# ── Aggregate SEDS to state-year-fuel level (Btu only) ───────────────────
seds_state_fuel = (
    seds_clean
    .filter(F.col("unit_type") == "B")
    .groupBy("year", "state_id", "state_name", "fuel_prefix")
    .agg(
        F.sum("value").alias("total_btu"),
        F.countDistinct("series_code").alias("num_series")
    )
)

# ── US annual totals by fuel prefix ───────────────────────────────────────
us_seds_total = (
    seds_state_fuel
    .groupBy("year", "fuel_prefix")
    .agg(F.sum("total_btu").alias("us_total_btu"))
)

# ── Join and compute state % of US for each fuel ──────────────────────────
seds_state_consumption = (
    seds_state_fuel
    .join(us_seds_total, on=["year", "fuel_prefix"], how="left")
    .withColumn(
        "pct_us_fuel_consumption",
        F.round(F.col("total_btu") / F.col("us_total_btu") * 100, 4)
    )
    .select(
        "year", "state_id", "state_name", "fuel_prefix",
        "total_btu", "us_total_btu", "pct_us_fuel_consumption", "num_series"
    )
    .orderBy("year", "state_id", "fuel_prefix")
)

print(f"SEDS state consumption rows: {seds_state_consumption.count():,}")
seds_state_consumption.show(10)

SEDS state consumption rows: 2,142
+----+--------+----------+-----------+---------+------------+-----------------------+----------+
|year|state_id|state_name|fuel_prefix|total_btu|us_total_btu|pct_us_fuel_consumption|num_series|
+----+--------+----------+-----------+---------+------------+-----------------------+----------+
|2024|      AK|    Alaska|         AV|   3288.0|     68316.0|                 4.8129|         3|
|2024|      AK|    Alaska|         B1|      0.0|   1462626.0|                    0.0|         5|
|2024|      AK|    Alaska|         BD|      0.0|   1393922.0|                    0.0|        17|
|2024|      AK|    Alaska|         BF|      0.0|   3615933.0|                    0.0|         2|
|2024|      AK|    Alaska|         BQ|      0.0|    378094.0|                    0.0|         2|
|2024|      AK|    Alaska|         BX|      0.0|    618155.0|                    0.0|         1|
|2024|      AK|    Alaska|         BY|      0.0|     -2776.0|                    0.0|       

### 4e. SEDS: Fuel Pivot — One Row per State-Year, Column per Fuel

In [23]:
fuel_prefixes = [
    r["fuel_prefix"] for r in
    seds_state_fuel.select("fuel_prefix").distinct().orderBy("fuel_prefix").collect()
]

seds_fuel_pivot = (
    seds_state_fuel
    .groupBy("year", "state_id", "state_name")
    .pivot("fuel_prefix", fuel_prefixes)
    .agg(F.first("total_btu"))
    .orderBy("year", "state_id")
)

# ── Compute each fuel's % of total (TE prefix = Total Energy) ─────────────
if "TE" in fuel_prefixes and "TE" in seds_fuel_pivot.columns:
    for prefix in fuel_prefixes:
        if prefix != "TE":
            seds_fuel_pivot = seds_fuel_pivot.withColumn(
                f"{prefix.lower()}_pct",
                F.round(F.col(f"`{prefix}`") / F.col("TE") * 100, 2)
            )

print(f"SEDS fuel pivot rows: {seds_fuel_pivot.count():,}")
seds_fuel_pivot.show(10)

SEDS fuel pivot rows: 51
+----+--------+--------------------+------+---------+--------+--------+---+--------+---+--------+---------+---------+------+--------+-------+--------+---+---------+---+--------+--------+--------+---+---+---------+------+---------+---------+---+--------+---------+---------+---------+-------+---+---+--------+-------+--------+--------+------+---------+---+--------+
|year|state_id|          state_name|    AV|       B1|      BD|      BF| BQ|      BX| BY|      CL|       DA|       DF|    DK|      DM|     EL|      EM| EQ|       ES| EY|      GE|      HL|      HY| IQ| IY|       JF|    KS|       MG|       MM| NA|      NC|       NG|       NN|       NU|     OH| PL| PP|      PQ|     PY|      RE|      RF|    SF|       SO| US|      WY|
+----+--------+--------------------+------+---------+--------+--------+---+--------+---+--------+---------+---------+------+--------+-------+--------+---+---------+---+--------+--------+--------+---+---+---------+------+---------+---------+---+-

### 4f. SEDS: State Energy Consumption as % of Total US Consumption (Annual)

In [24]:
# ── US annual total across all fuels (TE prefix) ──────────────────────────
us_seds_all = (
    seds_state_fuel
    .filter(F.col("fuel_prefix") == "TE")
    .groupBy("year")
    .agg(F.sum("total_btu").alias("us_all_btu"))
)

state_year_window = Window.partitionBy("state_id").orderBy("year")

# ── State total energy consumption + % of US + YoY ───────────────────────
seds_state_pct_us = (
    seds_state_fuel
    .filter(F.col("fuel_prefix") == "TE")
    .join(us_seds_all, on="year", how="left")
    .withColumn(
        "pct_us_total_consumption",
        F.round(F.col("total_btu") / F.col("us_all_btu") * 100, 4)
    )
    .withColumn("yoy_change_btu",
        F.col("total_btu") - F.lag("total_btu", 1).over(state_year_window)
    )
    .withColumn("yoy_change_pct",
        F.round(
            F.col("yoy_change_btu") / F.lag("total_btu", 1).over(state_year_window) * 100,
            2
        )
    )
    .select(
        "year", "state_id", "state_name",
        "total_btu", "us_all_btu",
        "pct_us_total_consumption",
        "yoy_change_btu", "yoy_change_pct"
    )
    .orderBy("year", F.col("total_btu").desc())
)

print(f"SEDS state % of US rows: {seds_state_pct_us.count():,}")
seds_state_pct_us.show(10)

SEDS state % of US rows: 0
+----+--------+----------+---------+----------+------------------------+--------------+--------------+
|year|state_id|state_name|total_btu|us_all_btu|pct_us_total_consumption|yoy_change_btu|yoy_change_pct|
+----+--------+----------+---------+----------+------------------------+--------------+--------------+
+----+--------+----------+---------+----------+------------------------+--------------+--------------+



### 4g. Petroleum Movements: State-Level Imports & Exports (Weekly)

In [25]:
# from pyspark.sql.window import Window
# Use actual process codes from the data (EEX = Exports, EIM = Imports)
MOVEMENT_PROCESSES = ["EEX", "EIM", "ENT"]

petro_import_export = (
    petro_movements_clean
    .filter(F.col("process_id").isin(MOVEMENT_PROCESSES))
    .groupBy("period", "duoarea", "area_name", "product_id", "product_name", "process_id")
    .agg(F.sum("value_kbd").alias("total_kbd"))
)

petro_movements_wide = (
    petro_import_export
    .groupBy("period", "duoarea", "area_name", "product_id", "product_name")
    .pivot("process_id", MOVEMENT_PROCESSES)
    .agg(F.first("total_kbd"))
    .withColumnRenamed("EEX", "exports_kbd")
    .withColumnRenamed("EIM", "imports_kbd")
    .withColumnRenamed("ENT", "net_imports_kbd")
    # ── Net trade balance: positive = net exporter ────────────────────────
    .withColumn(
        "trade_balance_kbd",
        F.col("exports_kbd") - F.col("imports_kbd")
    )
    .orderBy("period", "duoarea", "product_id")
)

# ── US weekly totals per product for % share ──────────────────────────────
us_petro_total = (
    petro_movements_clean
    .filter(F.col("duoarea") == "NUS-Z00")
    .filter(F.col("process_id").isin(MOVEMENT_PROCESSES))
    .groupBy("period", "process_id", "product_id")
    .agg(F.sum("value_kbd").alias("us_total_kbd"))
)

petro_by_area = (
    petro_import_export
    .join(us_petro_total, on=["period", "process_id", "product_id"], how="left")
    .withColumn(
        "pct_us",
        F.round(F.col("total_kbd") / F.col("us_total_kbd") * 100, 4)
    )
    .orderBy("period", "process_id", F.col("total_kbd").desc())
)

print(f"Petroleum movements wide rows: {petro_movements_wide.count():,}")
petro_movements_wide.show(10)
print(f"Petroleum by area rows: {petro_by_area.count():,}")
petro_by_area.show(10)

Petroleum movements wide rows: 10
+----------+-------+---------+----------+--------------------+-----------+-----------+---------------+-----------------+
|    period|duoarea|area_name|product_id|        product_name|exports_kbd|imports_kbd|net_imports_kbd|trade_balance_kbd|
+----------+-------+---------+----------+--------------------+-----------+-----------+---------------+-----------------+
|2025-08-01|NUS-Z00|     U.S.|      EP00|Crude Oil and Pet...|    10663.0|       NULL|           NULL|             NULL|
|2025-08-01|NUS-Z00|     U.S.|      EPC0|           Crude Oil|     3318.0|       NULL|           NULL|             NULL|
|2025-08-01|NUS-Z00|     U.S.|      EPD0| Distillate Fuel Oil|     1545.0|       NULL|           NULL|             NULL|
|2025-08-01|NUS-Z00|     U.S.|      EPJK|Kerosene-Type Jet...|      141.0|       NULL|           NULL|             NULL|
|2025-08-01|NUS-Z00|     U.S.|    EPLLPZ|Propane and Propy...|     1991.0|       NULL|           NULL|             NULL

### 4h. Electricity Generating Capacity: State Totals & % of US (Annual)

In [26]:
# ── US annual total capacity by energy source ─────────────────────────────
us_capacity_total = (
    elec_capacity_clean
    .groupBy("year", "energy_source")
    .agg(F.sum("capability_mw").alias("us_total_mw"))
)

# ── State capacity by source + % of US ───────────────────────────────────
elec_capacity_state = (
    elec_capacity_clean
    .groupBy("year", "state_id", "state_name", "energy_source")
    .agg(
        F.sum("capability_mw").alias("state_total_mw"),
        F.countDistinct("producer_type").alias("num_producer_types")
    )
    .join(us_capacity_total, on=["year", "energy_source"], how="left")
    .withColumn(
        "pct_us_capacity",
        F.round(F.col("state_total_mw") / F.col("us_total_mw") * 100, 4)
    )
    .select(
        "year", "state_id", "state_name", "energy_source",
        "state_total_mw", "us_total_mw", "pct_us_capacity",
        "num_producer_types"
    )
    .orderBy("year", "state_id", "energy_source")
)

print(f"Electricity capacity state rows: {elec_capacity_state.count():,}")
elec_capacity_state.show(10)

Electricity capacity state rows: 916
+----+--------+----------+----------------+-----------------+------------------+---------------+------------------+
|year|state_id|state_name|   energy_source|   state_total_mw|       us_total_mw|pct_us_capacity|num_producer_types|
+----+--------+----------+----------------+-----------------+------------------+---------------+------------------+
|2024|      AK|    Alaska|             All|           5706.8| 4921665.200000002|          0.116|                 3|
|2024|      AK|    Alaska|         Battery|            263.4|          108080.0|         0.2437|                 2|
|2024|      AK|    Alaska|            Coal|            335.8| 696710.3999999998|         0.0482|                 3|
|2024|      AK|    Alaska|   Hydroelectric|952.6000000000001| 319587.5999999998|         0.2981|                 3|
|2024|      AK|    Alaska|     Natural Gas|           2515.0|2025484.7999999993|         0.1242|                 3|
|2024|      AK|    Alaska|Natural G

## 5. Write Parquet to S3

**Requires a configured S3 bucket.** Fill in AWS credentials in Section 0 first.  
All cells above this point run entirely locally without AWS credentials.

In [27]:
def write_parquet(df, path, label):
    """Write a DataFrame to S3 as snappy-compressed Parquet."""
    print(f"Writing {label} → {path} ...")
    df.write.mode("overwrite").option("compression", "snappy").parquet(path)
    print(f"  ✓ Done")

# ── Raw / cleaned datasets ─────────────────────────────────────────────────
write_parquet(petroleum_clean,          S3_PETROLEUM_PRODUCTION,  "Petroleum Production (Cleaned)")
write_parquet(petro_movements_clean,    S3_PETROLEUM_MOVEMENTS,   "Petroleum Movements (Cleaned)")
write_parquet(coal_clean,               S3_COAL_TRADE,            "Coal Trade (Cleaned)")
write_parquet(elec_rankings_clean,      S3_ELECTRICITY_RANKINGS,  "Electricity State Rankings")
write_parquet(elec_net_meter_clean,     S3_ELECTRICITY_NET_METER, "Electricity Net Metering")
write_parquet(total_energy_clean,       S3_TOTAL_ENERGY,          "Total Energy (Cleaned)")

# ── Analytical datasets ────────────────────────────────────────────────────
write_parquet(elec_state_by_fuel,       S3_ELECTRICITY_BY_FUEL,   "Electricity by Fuel & State")
write_parquet(elec_capacity_state,      S3_ELECTRICITY_CAPACITY,  "Electricity Generating Capacity by State")
write_parquet(coal_trade_summary,       S3_COAL_TRADE + "summary/", "Coal Trade Summary w/ % of US")
write_parquet(seds_state_consumption,   S3_SEDS_STATE_CONSUMPTION,"SEDS State Consumption by Fuel")
write_parquet(seds_state_pct_us,        S3_SEDS_STATE_PCT,        "SEDS State % of US Total Energy")
write_parquet(seds_fuel_pivot,          S3_SEDS_FUEL_PIVOT,       "SEDS Fuel Pivot (Wide Format)")
write_parquet(petro_movements_wide,     S3_PETROLEUM_MOVEMENTS + "wide/", "Petroleum Movements Wide (Imp/Exp/Net)")

print("\nAll datasets written to S3 successfully.")

Writing Petroleum Production (Cleaned) → s3a://cs4266-eia-big-data-bucket/processed/petroleum_production/ ...
  ✓ Done
Writing Petroleum Movements (Cleaned) → s3a://cs4266-eia-big-data-bucket/processed/petroleum_movements/ ...
  ✓ Done
Writing Coal Trade (Cleaned) → s3a://cs4266-eia-big-data-bucket/processed/coal_trade/ ...
  ✓ Done
Writing Electricity State Rankings → s3a://cs4266-eia-big-data-bucket/processed/electricity_state_rankings/ ...
  ✓ Done
Writing Electricity Net Metering → s3a://cs4266-eia-big-data-bucket/processed/electricity_net_metering/ ...
  ✓ Done
Writing Total Energy (Cleaned) → s3a://cs4266-eia-big-data-bucket/processed/total_energy/ ...
  ✓ Done
Writing Electricity by Fuel & State → s3a://cs4266-eia-big-data-bucket/processed/electricity_by_fuel_state/ ...
  ✓ Done
Writing Electricity Generating Capacity by State → s3a://cs4266-eia-big-data-bucket/processed/electricity_generating_capacity/ ...
  ✓ Done
Writing Coal Trade Summary w/ % of US → s3a://cs4266-eia-big-da

## 6. Stop Spark

In [28]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
